# Student Performance Dataset

This notebook explores student GPA using academic, demographic, and support-related features. The goal is to keep each question readable: short text, compact code, table output, and a clear conclusion.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

csv_path = Path('Student_performance_data _.csv')
if not csv_path.exists():
    csv_path = Path('day2/Student_performance_data _.csv')

df = pd.read_csv(csv_path)
df.head()


,StudentID,Age,Gender,Ethnicity,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,Music,Volunteering,GPA,GradeClass
0,1001,17,1,0,2,19.833723,7,1,2,0,0,1,0,2.929196,2.0
1,1002,18,0,0,1,15.408756,0,0,1,0,0,0,0,3.042915,1.0
2,1003,15,0,2,3,4.210570,26,0,2,0,0,0,0,0.112602,4.0
3,1004,17,1,0,3,10.028829,14,0,3,1,0,0,0,2.054218,3.0
4,1005,17,1,0,2,4.672495,17,1,3,0,0,0,0,1.288061,4.0


## Basic Dataset Information

First, check the size of the dataset and the structure of the columns.


In [2]:
rows, columns = df.shape
print(f'Rows: {rows:,}')
print(f'Columns: {columns:,}')


Rows: 2,392
Columns: 15


In [3]:
column_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
    'missing_count': df.isna().sum().values,
    'missing_percent': (df.isna().mean() * 100).round(2).values,
    'unique_values': df.nunique().values
})

column_summary


,column,dtype,non_null_count,missing_count,missing_percent,unique_values
0,StudentID,int64,2392,0,0.0,2392
1,Age,int64,2392,0,0.0,4
2,Gender,int64,2392,0,0.0,2
3,Ethnicity,int64,2392,0,0.0,4
4,ParentalEducation,int64,2392,0,0.0,5
5,StudyTimeWeekly,float64,2392,0,0.0,2392
6,Absences,int64,2392,0,0.0,30
7,Tutoring,int64,2392,0,0.0,2
8,ParentalSupport,int64,2392,0,0.0,5
9,Extracurricular,int64,2392,0,0.0,2


In [4]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
StudentID,2392.0,2196.500000,690.655244,1001.000000,1598.750000,2196.500000,2794.250000,3392.000000
Age,2392.0,16.468645,1.123798,15.000000,15.000000,16.000000,17.000000,18.000000
Gender,2392.0,0.510870,0.499986,0.000000,0.000000,1.000000,1.000000,1.000000
Ethnicity,2392.0,0.877508,1.028476,0.000000,0.000000,0.000000,2.000000,3.000000
ParentalEducation,2392.0,1.746237,1.000411,0.000000,1.000000,2.000000,2.000000,4.000000
StudyTimeWeekly,2392.0,9.771992,5.652774,0.001057,5.043079,9.705363,14.408410,19.978094
Absences,2392.0,14.541388,8.467417,0.000000,7.000000,15.000000,22.000000,29.000000
Tutoring,2392.0,0.301421,0.458971,0.000000,0.000000,0.000000,1.000000,1.000000
ParentalSupport,2392.0,2.122074,1.122813,0.000000,1.000000,2.000000,3.000000,4.000000
Extracurricular,2392.0,0.383361,0.486307,0.000000,0.000000,0.000000,1.000000,1.000000


## Question 1: Does Higher Study Time Mean Better GPA?

Study time is grouped into 5-hour weekly ranges: 0-5, 5-10, 10-15, and 15-20 hours.


In [5]:
study_bins = [0, 5, 10, 15, 20]
study_labels = ['0-5', '5-10', '10-15', '15-20']

study_df = df.copy()
study_df['StudyTimeGroup'] = pd.cut(
    study_df['StudyTimeWeekly'],
    bins=study_bins,
    labels=study_labels,
    include_lowest=True,
    right=False
)

study_time_summary = (
    study_df.groupby('StudyTimeGroup', observed=True)
    .agg(
        student_count=('StudentID', 'count'),
        avg_study_hours=('StudyTimeWeekly', 'mean'),
        avg_gpa=('GPA', 'mean'),
        median_gpa=('GPA', 'median'),
        avg_grade_class=('GradeClass', 'mean')
    )
    .round(2)
    .reset_index()
)

study_correlation = study_df['StudyTimeWeekly'].corr(study_df['GPA'])
study_time_summary


,StudyTimeGroup,student_count,avg_study_hours,avg_gpa,median_gpa,avg_grade_class
0,0-5,595,2.48,1.69,1.68,3.19
1,5-10,643,7.52,1.85,1.81,3.05
2,10-15,619,12.41,2.00,2.04,2.89
3,15-20,535,17.53,2.11,2.12,2.77


**Conclusion:** GPA increases steadily as study time increases. The average GPA rises from **1.69** in the 0-5 hour group to **2.11** in the 15-20 hour group.

The relationship is positive but weak: the correlation between weekly study time and GPA is only **0.18**. Study time helps, but it does not explain grades by itself.


## Question 2: How Does Each Feature Group Relate to Grades?

For every feature, compare average GPA, median GPA, and average GradeClass by group. Study time and absences are grouped into ranges so their tables stay readable.


In [6]:
analysis_df = df.copy()
analysis_df['StudyTimeGroup'] = pd.cut(
    analysis_df['StudyTimeWeekly'],
    bins=[0, 5, 10, 15, 20],
    labels=['0-5', '5-10', '10-15', '15-20'],
    include_lowest=True,
    right=False
)
analysis_df['AbsencesGroup'] = pd.cut(
    analysis_df['Absences'],
    bins=[0, 5, 10, 15, 20, 25, 30],
    labels=['0-5', '5-10', '10-15', '15-20', '20-25', '25-30'],
    include_lowest=True,
    right=False
)

label_maps = {
    'Gender': {0: '0 - Female', 1: '1 - Male'},
    'Tutoring': {0: '0 - No', 1: '1 - Yes'},
    'Extracurricular': {0: '0 - No', 1: '1 - Yes'},
    'Sports': {0: '0 - No', 1: '1 - Yes'},
    'Music': {0: '0 - No', 1: '1 - Yes'},
    'Volunteering': {0: '0 - No', 1: '1 - Yes'},
}

feature_specs = [
    ('Age', 'Age'),
    ('Gender', 'Gender'),
    ('Ethnicity', 'Ethnicity'),
    ('ParentalEducation', 'ParentalEducation'),
    ('StudyTimeWeekly', 'StudyTimeGroup'),
    ('Absences', 'AbsencesGroup'),
    ('Tutoring', 'Tutoring'),
    ('ParentalSupport', 'ParentalSupport'),
    ('Extracurricular', 'Extracurricular'),
    ('Sports', 'Sports'),
    ('Music', 'Music'),
    ('Volunteering', 'Volunteering'),
]

summary_tables = []
for feature, group_col in feature_specs:
    grouped = (
        analysis_df.groupby(group_col, observed=True)
        .agg(
            student_count=('StudentID', 'count'),
            avg_gpa=('GPA', 'mean'),
            median_gpa=('GPA', 'median'),
            avg_grade_class=('GradeClass', 'mean')
        )
        .reset_index()
        .rename(columns={group_col: 'group'})
    )
    if feature in label_maps:
        grouped['group'] = grouped['group'].map(label_maps[feature])
    elif group_col == feature:
        grouped['group'] = grouped['group'].apply(lambda value: f'{feature} {int(value)}')
    grouped.insert(0, 'feature', feature)
    summary_tables.append(grouped)

feature_group_summary = pd.concat(summary_tables, ignore_index=True)
feature_group_summary[['avg_gpa', 'median_gpa', 'avg_grade_class']] = (
    feature_group_summary[['avg_gpa', 'median_gpa', 'avg_grade_class']].round(2)
)

feature_effect_summary = []
for feature in feature_group_summary['feature'].unique():
    rows_for_feature = feature_group_summary[feature_group_summary['feature'] == feature]
    highest = rows_for_feature.loc[rows_for_feature['avg_gpa'].idxmax()]
    lowest = rows_for_feature.loc[rows_for_feature['avg_gpa'].idxmin()]
    feature_effect_summary.append({
        'feature': feature,
        'highest_avg_gpa_group': highest['group'],
        'highest_avg_gpa': highest['avg_gpa'],
        'lowest_avg_gpa_group': lowest['group'],
        'lowest_avg_gpa': lowest['avg_gpa'],
        'gpa_gap': round(highest['avg_gpa'] - lowest['avg_gpa'], 2)
    })

feature_effect_summary = (
    pd.DataFrame(feature_effect_summary)
    .sort_values('gpa_gap', ascending=False)
    .reset_index(drop=True)
)

display(feature_group_summary)
display(feature_effect_summary)


,feature,group,student_count,avg_gpa,median_gpa,avg_grade_class
0,Age,Age 15,630,1.90,1.89,3.00
1,Age,Age 16,593,1.91,1.89,2.98
2,Age,Age 17,587,1.93,1.99,2.98
3,Age,Age 18,582,1.89,1.80,2.97
4,Gender,0 - Female,1170,1.92,1.92,2.95
5,Gender,1 - Male,1222,1.89,1.88,3.01
6,Ethnicity,Ethnicity 0,1207,1.88,1.84,3.02
7,Ethnicity,Ethnicity 1,493,1.95,1.95,2.94
8,Ethnicity,Ethnicity 2,470,1.92,1.88,2.96
9,Ethnicity,Ethnicity 3,222,1.95,2.00,2.94


,feature,highest_avg_gpa_group,highest_avg_gpa,lowest_avg_gpa_group,lowest_avg_gpa,gpa_gap
0,Absences,0-5,3.16,25-30,0.69,2.47
1,ParentalSupport,ParentalSupport 4,2.19,ParentalSupport 0,1.54,0.65
2,StudyTimeWeekly,15-20,2.11,0-5,1.69,0.42
3,Tutoring,1 - Yes,2.11,0 - No,1.82,0.29
4,Extracurricular,1 - Yes,2.02,0 - No,1.84,0.18
5,Music,1 - Yes,2.04,0 - No,1.87,0.17
6,ParentalEducation,ParentalEducation 1,1.94,ParentalEducation 3,1.81,0.13
7,Sports,1 - Yes,1.99,0 - No,1.87,0.12
8,Ethnicity,Ethnicity 1,1.95,Ethnicity 0,1.88,0.07
9,Age,Age 17,1.93,Age 18,1.89,0.04


**Conclusion:** Absences have the biggest difference in GPA by far. Students with 0-5 absences average **3.16** GPA, while students with 25-30 absences average **0.69** GPA. That is a **2.47 GPA point gap**.

After absences, the strongest differences are ParentalSupport (**0.65** gap), StudyTimeWeekly (**0.42** gap), and Tutoring (**0.29** gap). Gender, age, ethnicity, and volunteering show much smaller differences.


## Question 3: Can High Study Time Compensate for Many Absences?

This checks the trap from Question 1. Study time looks helpful, but what happens when students miss many classes?


In [7]:
interaction_df = df.copy()
interaction_df['StudyTimeGroup'] = pd.cut(
    interaction_df['StudyTimeWeekly'],
    bins=[0, 5, 10, 15, 20],
    labels=['0-5', '5-10', '10-15', '15-20'],
    include_lowest=True,
    right=False
)
interaction_df['AbsencesGroup'] = pd.cut(
    interaction_df['Absences'],
    bins=[0, 5, 10, 15, 20, 25, 30],
    labels=['0-5', '5-10', '10-15', '15-20', '20-25', '25-30'],
    include_lowest=True,
    right=False
)

avg_gpa_by_absence_and_study = interaction_df.pivot_table(
    index='AbsencesGroup',
    columns='StudyTimeGroup',
    values='GPA',
    aggfunc='mean',
    observed=True
).round(2)

student_count_check = interaction_df.pivot_table(
    index='AbsencesGroup',
    columns='StudyTimeGroup',
    values='StudentID',
    aggfunc='count',
    observed=True
).astype(int)

comparison_filters = {
    'Low study, low absences': ('0-5', '0-5'),
    'High study, low absences': ('15-20', '0-5'),
    'High study, high absences': ('15-20', '25-30'),
}

key_groups = []
for name, (study_group, absence_group) in comparison_filters.items():
    filtered = interaction_df[
        (interaction_df['StudyTimeGroup'] == study_group)
        & (interaction_df['AbsencesGroup'] == absence_group)
    ]
    key_groups.append({
        'comparison_group': name,
        'study_hours_group': study_group,
        'absences_group': absence_group,
        'student_count': len(filtered),
        'avg_gpa': filtered['GPA'].mean()
    })

key_groups = pd.DataFrame(key_groups).round({'avg_gpa': 2})

display(avg_gpa_by_absence_and_study)
display(student_count_check)
display(key_groups)


StudyTimeGroup,0-5,5-10,10-15,15-20
AbsencesGroup,,,,
0-5,2.98,3.11,3.20,3.38
5-10,2.43,2.59,2.72,2.88
10-15,1.97,2.07,2.19,2.41
15-20,1.46,1.58,1.78,1.90
20-25,0.95,1.09,1.25,1.35
25-30,0.46,0.60,0.76,0.96


StudyTimeGroup,0-5,5-10,10-15,15-20
AbsencesGroup,,,,
0-5,85,104,102,78
5-10,105,113,109,88
10-15,97,101,114,89
15-20,112,120,91,93
20-25,107,105,98,101
25-30,89,100,105,86


,comparison_group,study_hours_group,absences_group,student_count,avg_gpa
0,"Low study, low absences",0-5,0-5,85,2.98
1,"High study, low absences",15-20,0-5,78,3.38
2,"High study, high absences",15-20,25-30,86,0.96


**Conclusion:** High study time does not compensate for very high absences.

Students who study 15-20 hours but have 25-30 absences average only **0.96** GPA. Students who study only 0-5 hours but have 0-5 absences average **2.98** GPA. The high-study/high-absence group is **2.02 GPA points lower**, or about **67.8% lower**.

**Check:** This is not a tiny-sample artifact. The smallest cell in the study-time by absences table still has **78 students**.


## Conclusion Check

The first three conclusions still hold after checking the numbers.

| Question | Main result | Check |
|---|---:|---|
| Q1 | Study time correlation with GPA is 0.18 | Positive but weak |
| Q2 | Absences have the largest GPA gap | 2.47 GPA points |
| Q3 | High study cannot rescue high absences | Smallest table cell has 78 students |


## Question 4: What Is the Hidden Trap in the Study-Time Correlation?

The raw correlation says study time is weak. This checks whether absences are hiding part of the study-time signal.


In [8]:
coeffs = np.polyfit(df['Absences'], df['GPA'], 1)
gpa_residual = df['GPA'] - np.polyval(coeffs, df['Absences'])
r2_absences = 1 - (gpa_residual**2).sum() / ((df['GPA'] - df['GPA'].mean())**2).sum()

numeric_features = [
    'StudyTimeWeekly', 'ParentalSupport', 'Tutoring', 'Extracurricular',
    'Music', 'Sports', 'ParentalEducation', 'Gender', 'Volunteering', 'Age'
]

comparison = []
for feature in numeric_features:
    raw_r = df[feature].corr(df['GPA'])
    net_r = df[feature].corr(gpa_residual)
    comparison.append({
        'feature': feature,
        'raw_r_with_gpa': round(raw_r, 3),
        'net_r_after_removing_absences': round(net_r, 3),
        'multiplier': round(abs(net_r / raw_r), 1) if raw_r != 0 else np.nan
    })

comparison = (
    pd.DataFrame(comparison)
    .sort_values('net_r_after_removing_absences', key=abs, ascending=False)
    .reset_index(drop=True)
)

print(f'R-squared from absences alone: {r2_absences:.3f}')
comparison


R-squared from absences alone: 0.845


,feature,raw_r_with_gpa,net_r_after_removing_absences,multiplier
0,ParentalSupport,0.191,0.490,2.6
1,StudyTimeWeekly,0.179,0.477,2.7
2,Tutoring,0.145,0.332,2.3
3,Sports,0.058,0.244,4.2
4,Extracurricular,0.094,0.240,2.6
5,Music,0.073,0.166,2.3
6,Volunteering,0.003,-0.035,10.7
7,Age,0.000,-0.026,95.1
8,Gender,-0.013,0.016,1.2
9,ParentalEducation,-0.036,-0.006,0.2


**Conclusion:** The statement "study time barely matters" is technically true in the raw correlation, but it is misleading.

Absences alone explain **84.5%** of GPA variance. After removing the absences effect, StudyTimeWeekly has a net correlation of **0.477** with the remaining GPA variation. That is about **2.7x** larger than the raw correlation of **0.179**.

A better wording is: study time looks weak when absences are mixed in, but it has a moderate relationship with GPA after accounting for absences.


## Mini-Investigation: Five Precise Facts

This section collects the strongest findings into short, presentation-ready facts.


In [9]:
r_abs = df['Absences'].corr(df['GPA'])
r_study = df['StudyTimeWeekly'].corr(df['GPA'])
ratio = abs(r_abs / r_study)

coeffs = np.polyfit(df['Absences'], df['GPA'], 1)
cost_per_absence = abs(coeffs[0])

best = df[(df['Absences'] <= 5) & (df['Tutoring'] == 1) & (df['StudyTimeWeekly'] >= 15)]
worst = df[(df['Absences'] >= 25) & (df['Tutoring'] == 0) & (df['StudyTimeWeekly'] < 5)]
rescue = df[(df['Absences'] >= 25) & (df['Tutoring'] == 1) & (df['StudyTimeWeekly'] >= 15)]
attend_only = df[df['Absences'] <= 5]

parental_education_ci = df.groupby('ParentalEducation').agg(
    n=('GPA', 'count'),
    mean_gpa=('GPA', 'mean'),
    std_gpa=('GPA', 'std')
)
parental_education_ci['ci95_low'] = (
    parental_education_ci['mean_gpa']
    - 1.96 * parental_education_ci['std_gpa'] / parental_education_ci['n']**0.5
)
parental_education_ci['ci95_high'] = (
    parental_education_ci['mean_gpa']
    + 1.96 * parental_education_ci['std_gpa'] / parental_education_ci['n']**0.5
)

mini_facts = pd.DataFrame([
    ['Absences vs study time', f'Absences are {ratio:.1f}x stronger by raw correlation'],
    ['Cost per absence', f'About {cost_per_absence:.2f} GPA points per absence'],
    ['Best profile', f'GPA {best["GPA"].mean():.2f}, n={len(best)}'],
    ['Worst profile', f'GPA {worst["GPA"].mean():.2f}, n={len(worst)}'],
    ['High-absence rescue group', f'GPA {rescue["GPA"].mean():.2f}, n={len(rescue)}'],
], columns=['finding', 'number'])

display(mini_facts)
display(parental_education_ci[['n', 'mean_gpa', 'ci95_low', 'ci95_high']].round(3))


,finding,number
0,Absences vs study time,Absences are 5.1x stronger by raw correlation
1,Cost per absence,About 0.10 GPA points per absence
2,Best profile,"GPA 3.48, n=33"
3,Worst profile,"GPA 0.42, n=67"
4,High-absence rescue group,"GPA 1.12, n=28"


,n,mean_gpa,ci95_low,ci95_high
ParentalEducation,,,,
0,243,1.893,1.780,2.006
1,728,1.944,1.879,2.009
2,934,1.930,1.870,1.990
3,367,1.809,1.712,1.906
4,120,1.816,1.671,1.961


**Conclusion:** The best 30-second finding is still the same trap: study time matters, but absence dominates the dataset.

The cleanest line is: "Study time looks weak at first, but that is because absences explain 84.5% of GPA variance. Control for absences first, or every other signal looks smaller than it really is."
